```markdown
>[!INFO]
> Mimo iż praca jest pisana w języku Polskim, z racji na konwencje przyjęte w branży, całość kodu zapisana jest w języku angielskim
```

# Benchmark techniczny modeli detekcji obiektów dla zadania pomiaru ruchu drogowego

Celem notebooka jest porównanie wybranych modeli detekcji obiektów pod kątem ich przydatności jako pierwszego etapu systemu pomiaru ruchu drogowego z nagrań wideo. Na tym etapie oceniane są: poprawność techniczna uruchomienia modelu, czas inferencji, liczba wykrytych obiektów należących do klas ruchu drogowego oraz możliwość ujednolicenia wyników do wspólnego formatu.

Benchmark wykorzystuje obrazy oraz maski instancji ze zbioru Mapillary. Maski instancji służą do obliczenia referencyjnej liczby obiektów w klasach istotnych dla projektu. Wyniki modeli są następnie porównywane z tymi wartościami referencyjnymi.

Notebook nie jest pełną ewaluacją detekcji w rozumieniu metryk IoU/mAP, ponieważ nie sprawdza dokładności położenia ramek ograniczających. Jest to benchmark przydatności modeli do zadania zliczania obiektów ruchu drogowego oraz wyboru modelu do dalszej integracji z modułami śledzenia i kategoryzacji.

---


### Zestawienie modeli detekcji wziętych pod uwagę

| Nazwa modelu | Architektura modelu | Właściciel praw | Licencja | Dataset treningowy | AP (wg. oficjalnego źródła) |
| -- | -- | -- | -- | -- | -- |
| fasterrcnn_resnet50_fpn | CNN (ResNet-50 + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 37.0 |
| fasterrcnn_mobilenet_v3_large_fpn | CNN (MobileNetV3-Large + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 32.8 |
| retinanet_resnet50_fpn | CNN (ResNet-50 + FPN, RetinaNet) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 36.4 |
| retinanet_resnet50_fpn_v2 | CNN (ResNet-50 + FPN, RetinaNet v2) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 41.5 |
| fcos_resnet50_fpn | CNN (ResNet-50 + FPN, anchor-free FCOS) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 39.2 |
| ssd300_vgg16 | CNN (VGG16, SSD300) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 25.1 |
| ssdlite320_mobilenet_v3_large | CNN (MobileNetV3-Large, SSDLite320) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 21.3 |
| facebook/detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Facebook / Meta AI| Apache-2.0 | COCO 2017 | 42.0 |
| microsoft/conditional-detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Microsoft | Apache-2.0 | COCO 2017 | — |
| SenseTime/deformable-detr | Hybrydowa (CNN backbone + Transformer) | SenseTime | Apache-2.0 | COCO 2017 | — |
| PekingU/rtdetr_r18vd | Hybrydowa (CNN backbone + Transformer) | Peking University (PekingU) | Apache-2.0 | COCO train2017 | 46.5 |

### Przygotowanie środowiska

In [15]:
# Import all neccessary libraries
import os
import sys
import time
import json
import psutil
import torch
import numpy as np
import pandas as pd
import PIL.Image as Image
from pathlib import Path
from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

# silence huggingface logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
disable_progress_bars()

# silence transformers logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

# import pyvision models
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, fasterrcnn_mobilenet_v3_large_fpn,
    retinanet_resnet50_fpn, retinanet_resnet50_fpn_v2,
    fcos_resnet50_fpn, ssd300_vgg16, ssdlite320_mobilenet_v3_large
)
from torchvision.models import list_models, get_model, get_model_weights
# import transformers utils
from transformers import AutoModelForObjectDetection, AutoImageProcessor

# mount google drive test data
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

MAPILLARY_TEST_IMAGE = Path('/content/drive/MyDrive/colab/mapilary_test_data/')
REAL_LIFE_TEST_VIDEO = Path('/content/drive/MyDrive/colab/test_data/irl_video/')

INSTANCE_PATH = Path(MAPILLARY_TEST_IMAGE/"instances")
IMAGE_PATH = Path(MAPILLARY_TEST_IMAGE/"images")
CONFIG_PATH = MAPILLARY_TEST_IMAGE / "config.json"
RESULTS_PATH = MAPILLARY_TEST_IMAGE / "results"
RESULTS_PATH.mkdir(exist_ok=True)


Mounted at /content/drive/


In [16]:
# define constant model list
MODELS = [
    "fasterrcnn_resnet50_fpn",
    "fasterrcnn_mobilenet_v3_large_fpn",
    "retinanet_resnet50_fpn",
    "retinanet_resnet50_fpn_v2",
    "fcos_resnet50_fpn",
    "ssd300_vgg16",
    "ssdlite320_mobilenet_v3_large",
    "facebook/detr-resnet-50",
    "microsoft/conditional-detr-resnet-50",
    "SenseTime/deformable-detr",
    "PekingU/rtdetr_r18vd"
]

TV_MODELS = set(list_models())
# define test environment

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG: Using device: {device}")

DEBUG: Using device: cuda


In [17]:
# loading model function
def load_model(model_name, device):
    if model_name in TV_MODELS:
        # print(f"DEBUG: Loading torchvision model: {model_name}")
        weights = get_model_weights(model_name).DEFAULT
        model = get_model(model_name, weights=weights).to(device).eval()
        preprocessing = weights.transforms()

        meta = {
            "model_name": model_name,
            "type": "torchvision",
            "weights": str(weights),
            "categories": weights.meta.get("categories"),
        }
        return model, preprocessing, meta
    
    # print(f"DEBUG: Loading transformers model: {model_name}")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForObjectDetection.from_pretrained(model_name).to(device).eval()
    meta = {
        "model_name": model_name,
        "type": "transformers",
        "weights": "from pretrained",
        "categories": model.config.id2label,
    }
    return model, processor, meta

# look for meta parameters in the model metadata
# this is to check if the metadata contains the necessary information for benchmarking

def check_meta_parameters(model, model_name):
    meta_params = [
        name for name, param in model.named_parameters()
        if getattr(param, "is_meta", False)
    ]

    meta_buffers = [
        name for name, buffer in model.named_buffers()
        if getattr(buffer, "is_meta", False)
    ]

    if meta_params or meta_buffers:
        print(f"[WARNING] {model_name}: model still has meta tensors")
        print(f"Meta parameters: {len(meta_params)}")
        print(f"Meta buffers: {len(meta_buffers)}")
        print("First examples:", meta_params[:10])
        return False

    print(f"[OK] {model_name}: no meta tensors detected")
    return True 

In [18]:
# loding model and logging metadata

loaded_models = {}

for n, model_name in enumerate(MODELS, start=1):
    try:
        model, processor, metadata = load_model(model_name, device)
        print(f"[{n}/{len(MODELS)}] Loaded {model_name} \nwith metadata: {metadata}")
        check_meta_parameters(model, model_name)
        loaded_models[model_name] = (model, processor, metadata)
    except Exception as e:
        print(f"ERROR while loading {model_name}: {e}")

[1/11] Loaded fasterrcnn_resnet50_fpn 
with metadata: {'model_name': 'fasterrcnn_resnet50_fpn', 'type': 'torchvision', 'weights': 'FasterRCNN_ResNet50_FPN_Weights.COCO_V1', 'categories': ['__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'micr

### Logika detekcji

In [19]:
# define objects of interest for benchmarking and select their corresponding class ids from the model metadata

CLASSES = ["pedestrian", "bicycle", "motorcycle", "vehicle"]

# class names used by COCO-like object detection models mapped to categories used in this project
DETECTION_LIBRARY = {
    "person" : "pedestrian",
    "pedestrian" : "pedestrian",
    "bicycle" : "bicycle",
    "bike" : "bicycle",
    "car" : "vehicle",
    "bus" : "vehicle",
    "truck" : "vehicle",
    "motorcycle" : "motorcycle",
    "motorbike" :  "motorcycle",
}

OBJECTS = list(DETECTION_LIBRARY.keys())

def normalise_class_name(class_name):
    return str(class_name).lower().strip()

def get_category_from_class_name(class_name):
    return DETECTION_LIBRARY.get(normalise_class_name(class_name))

def get_object_id(metadata, objects=OBJECTS):
    model_name = metadata["model_name"]
    categories = metadata["categories"]
    selected_ids = {}

    if isinstance(categories, list):
        iterable = enumerate(categories)
    elif isinstance(categories, dict):
        iterable = categories.items()
    else:
        print(f"[ERROR] {model_name}: unknown categories format: {type(categories)}")
        return selected_ids

    for class_id, class_name in iterable:
        class_id = int(class_id)
        class_name = normalise_class_name(class_name)
        category = get_category_from_class_name(class_name)
        if category is not None:
            selected_ids[class_id] = {
                "class_name": class_name,
                "category": category,
            }
    return selected_ids

# Sanity check: every loaded model should expose at least some classes selected for the benchmark.
# If a model prints an empty list, its label mapping needs to be inspected before trusting the result.
for model_name, model_bundle in loaded_models.items():
    print(model_name, get_object_id(model_bundle[2]))


fasterrcnn_resnet50_fpn {1: {'class_name': 'person', 'category': 'pedestrian'}, 2: {'class_name': 'bicycle', 'category': 'bicycle'}, 3: {'class_name': 'car', 'category': 'vehicle'}, 4: {'class_name': 'motorcycle', 'category': 'motorcycle'}, 6: {'class_name': 'bus', 'category': 'vehicle'}, 8: {'class_name': 'truck', 'category': 'vehicle'}}
fasterrcnn_mobilenet_v3_large_fpn {1: {'class_name': 'person', 'category': 'pedestrian'}, 2: {'class_name': 'bicycle', 'category': 'bicycle'}, 3: {'class_name': 'car', 'category': 'vehicle'}, 4: {'class_name': 'motorcycle', 'category': 'motorcycle'}, 6: {'class_name': 'bus', 'category': 'vehicle'}, 8: {'class_name': 'truck', 'category': 'vehicle'}}
retinanet_resnet50_fpn {1: {'class_name': 'person', 'category': 'pedestrian'}, 2: {'class_name': 'bicycle', 'category': 'bicycle'}, 3: {'class_name': 'car', 'category': 'vehicle'}, 4: {'class_name': 'motorcycle', 'category': 'motorcycle'}, 6: {'class_name': 'bus', 'category': 'vehicle'}, 8: {'class_name': '

In [20]:
# define method for object detection on still images

def detect_from_image(model_name, model_bundle, device, image_path, threshold=0.5):
    curr_model, processor, metadata = model_bundle
    object_ids = get_object_id(metadata)

    image = Image.open(image_path).convert("RGB")
    detections = []

    if metadata["type"] == "torchvision":
        processed_image = processor(image).to(device)
        with torch.inference_mode():
            prediction = curr_model([processed_image])[0]

        for box, label_id, score in zip(
            prediction["boxes"],
            prediction["labels"],
            prediction["scores"],
        ):
            label_id = int(label_id)
            score = float(score)

            if label_id not in object_ids:
                continue
            if score < threshold:
                continue

            detections.append({
                "class_id": label_id,
                "class_name": object_ids[label_id]["class_name"],
                "category": object_ids[label_id]["category"],
                "score": score,
                "box": box.detach().cpu().tolist(),
            })
    else:
        inputs = processor(images=image, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.inference_mode():
            outputs = curr_model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]], device=device)
        results = processor.post_process_object_detection(
            outputs,
            threshold=threshold,
            target_sizes=target_sizes,
        )[0]

        for box, label_id, score in zip(
            results["boxes"],
            results["labels"],
            results["scores"],
        ):
            label_id = int(label_id)
            score = float(score)

            if label_id not in object_ids:
                continue

            detections.append({
                "class_id": label_id,
                "class_name": object_ids[label_id]["class_name"],
                "category": object_ids[label_id]["category"],
                "score": score,
                "box": box.detach().cpu().tolist(),
            })

    return detections

def count_detections_by_category(detections):
    counts = {category: 0 for category in CLASSES}
    for detection in detections:
        category = detection["category"]
        if category in counts:
            counts[category] += 1
    counts["total"] = sum(counts.values())
    return counts


## Opracowanie danych kontrolnych

In [21]:
import json

with open(CONFIG_PATH, "r") as f:
    mapillary_config = json.load(f)

# Mapillary uses detailed classes. For this benchmark they are grouped into the same
# categories as model detections, so counts can be compared directly.
MAPILLARY_TARGET_NAMES = {
    "human--person": "pedestrian",

    "object--vehicle--bicycle": "bicycle",
    "object--vehicle--bus": "vehicle",
    "object--vehicle--car": "vehicle",
    "object--vehicle--caravan": "vehicle",
    "object--vehicle--motorcycle": "motorcycle",
    "object--vehicle--other-vehicle": "vehicle",
    "object--vehicle--trailer": "vehicle",
    "object--vehicle--truck": "vehicle",
    "object--vehicle--wheeled-slow": "vehicle",
}

def build_mapillary_target_ids(config, target_names):
    target_ids = {}

    for class_id, label in enumerate(config["labels"]):
        label_name = label["name"]

        if label_name in target_names:
            target_ids[class_id] = {
                "readable": label["readable"],
                "name": label["name"],
                "target_class": target_names[label_name],
                "instances": label["instances"],
                "evaluate": label["evaluate"],
            }

    return target_ids

MAPILLARY_TARGET_IDS = build_mapillary_target_ids(
    mapillary_config,
    MAPILLARY_TARGET_NAMES,
)

MAPILLARY_TARGET_IDS


{19: {'readable': 'Person',
  'name': 'human--person',
  'target_class': 'pedestrian',
  'instances': True,
  'evaluate': True},
 52: {'readable': 'Bicycle',
  'name': 'object--vehicle--bicycle',
  'target_class': 'bicycle',
  'instances': True,
  'evaluate': True},
 54: {'readable': 'Bus',
  'name': 'object--vehicle--bus',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 55: {'readable': 'Car',
  'name': 'object--vehicle--car',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 56: {'readable': 'Caravan',
  'name': 'object--vehicle--caravan',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 57: {'readable': 'Motorcycle',
  'name': 'object--vehicle--motorcycle',
  'target_class': 'motorcycle',
  'instances': True,
  'evaluate': True},
 59: {'readable': 'Other Vehicle',
  'name': 'object--vehicle--other-vehicle',
  'target_class': 'vehicle',
  'instances': True,
  'evaluate': True},
 60: {'readable': 'Trailer',
  'name':

In [22]:
print(MAPILLARY_TEST_IMAGE)
print(IMAGE_PATH.exists())
print(INSTANCE_PATH.exists())
print(CONFIG_PATH.exists())

print("images:", len(list(IMAGE_PATH.glob("*"))))
print("instances:", len(list(INSTANCE_PATH.glob("*"))))

/content/drive/MyDrive/colab/mapilary_test_data
True
True
True
images: 20
instances: 20


In [23]:
# test does instance mask exists
def find_instance_mask(img_name):
    img_path = IMAGE_PATH / img_name
    instance_path = INSTANCE_PATH / f"{img_path.stem}.png"

    if instance_path.exists():
        return instance_path

    return None


In [24]:
# count objects from the instace mask

def count_obj_from_instance(instance_path):
    mask = np.array(Image.open(instance_path))

    baseline_count = {category: 0 for category in CLASSES}

    for instance_value in np.unique(mask):
        if instance_value == 0:
            continue
        class_id = int(instance_value) // 256
        if class_id not in MAPILLARY_TARGET_IDS:
            continue
        target_class = MAPILLARY_TARGET_IDS[class_id]["target_class"]
        baseline_count[target_class] += 1

    baseline_count["total"] = sum(baseline_count.values())

    return baseline_count


In [25]:
# Create DataFrame of reference data for all images

def list_test_images():
    return sorted([
        image_name for image_name in os.listdir(IMAGE_PATH)
        if Path(image_name).suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

def build_reference_data(image_names):
    rows = []
    for image_name in image_names:
        image_key = Path(image_name).stem
        instance_path = find_instance_mask(image_name)

        if instance_path is None:
            print(f"[WARNING] Missing instance mask for {image_name}")
            continue

        counts = count_obj_from_instance(instance_path)
        row = {"image": image_key}
        for category in CLASSES:
            row[f"{category}_gt"] = counts[category]
        row["total_gt"] = counts["total"]
        rows.append(row)

    return pd.DataFrame(rows)

TEST_IMAGES = list_test_images()
REFERENCE_DATA = build_reference_data(TEST_IMAGES)
REFERENCE_DATA.to_csv(RESULTS_PATH / "reference_counts.csv", index=False)

print(f"Reference images: {len(REFERENCE_DATA)} / {len(TEST_IMAGES)}")
REFERENCE_DATA.head()


Reference images: 20 / 20


,image,pedestrian_gt,bicycle_gt,motorcycle_gt,vehicle_gt,total_gt
0,_1Gn_xkw7sa_i9GU4mkxxQ,1,0,4,5,10
1,_2yFuPMD1eo4D1sHkx-c4Q,0,0,0,5,5
2,_3JWmnl2ILeemJ9qoxHJCg,0,0,0,8,8
3,_4SjTmQ-zn3XSv4D1-Tg4w,18,0,0,17,35
4,_4ycMRRZcIpTeDT8MqtIDw,0,0,0,13,13


In [26]:
# Run detection benchmark and save model predictions in count format

THRESHOLDS = [0.3, 0.5, 0.7]

def detection_runner(image_names=TEST_IMAGES, thresholds=THRESHOLDS):
    prediction_rows = []
    raw_detection_rows = []

    for threshold in thresholds:
        print(f"\n=== Threshold: {threshold} ===")

        for model_name, model_bundle in loaded_models.items():
            print(f"Running model: {model_name}")

            for image_name in image_names:
                image_key = Path(image_name).stem
                image_path = IMAGE_PATH / image_name

                if device == "cuda":
                    torch.cuda.synchronize()
                start_time = time.perf_counter()
                detections = detect_from_image(
                    model_name=model_name,
                    model_bundle=model_bundle,
                    device=device,
                    image_path=image_path,
                    threshold=threshold,
                )
                if device == "cuda":
                    torch.cuda.synchronize()
                inference_time = time.perf_counter() - start_time

                counts = count_detections_by_category(detections)

                prediction_row = {
                    "image": image_key,
                    "model": model_name,
                    "threshold": threshold,
                    "inference_time_s": inference_time,
                    "fps": 1 / inference_time if inference_time > 0 else np.nan,
                }
                for category in CLASSES:
                    prediction_row[f"{category}_pred"] = counts[category]
                prediction_row["total_pred"] = counts["total"]
                prediction_rows.append(prediction_row)

                for detection in detections:
                    raw_detection_rows.append({
                        "image": image_key,
                        "model": model_name,
                        "threshold": threshold,
                        "class_name": detection["class_name"],
                        "category": detection["category"],
                        "score": detection["score"],
                        "box": detection["box"],
                    })

    prediction_counts = pd.DataFrame(prediction_rows)
    raw_detections = pd.DataFrame(raw_detection_rows)
    return prediction_counts, raw_detections

PREDICTION_COUNTS, RAW_DETECTIONS = detection_runner()

PREDICTION_COUNTS.to_csv(RESULTS_PATH / "prediction_counts.csv", index=False)
RAW_DETECTIONS.to_csv(RESULTS_PATH / "raw_detections.csv", index=False)

PREDICTION_COUNTS.head()



=== Threshold: 0.3 ===
Running model: fasterrcnn_resnet50_fpn
Running model: fasterrcnn_mobilenet_v3_large_fpn
Running model: retinanet_resnet50_fpn
Running model: retinanet_resnet50_fpn_v2
Running model: fcos_resnet50_fpn
Running model: ssd300_vgg16
Running model: ssdlite320_mobilenet_v3_large
Running model: facebook/detr-resnet-50
Running model: microsoft/conditional-detr-resnet-50
Running model: SenseTime/deformable-detr
Running model: PekingU/rtdetr_r18vd

=== Threshold: 0.5 ===
Running model: fasterrcnn_resnet50_fpn
Running model: fasterrcnn_mobilenet_v3_large_fpn
Running model: retinanet_resnet50_fpn
Running model: retinanet_resnet50_fpn_v2
Running model: fcos_resnet50_fpn
Running model: ssd300_vgg16
Running model: ssdlite320_mobilenet_v3_large
Running model: facebook/detr-resnet-50
Running model: microsoft/conditional-detr-resnet-50
Running model: SenseTime/deformable-detr
Running model: PekingU/rtdetr_r18vd

=== Threshold: 0.7 ===
Running model: fasterrcnn_resnet50_fpn
Running

,image,model,threshold,inference_time_s,fps,pedestrian_pred,bicycle_pred,motorcycle_pred,vehicle_pred,total_pred
0,_1Gn_xkw7sa_i9GU4mkxxQ,fasterrcnn_resnet50_fpn,0.3,0.386411,2.587918,6,0,3,6,15
1,_2yFuPMD1eo4D1sHkx-c4Q,fasterrcnn_resnet50_fpn,0.3,0.336866,2.968542,2,0,0,6,8
2,_3JWmnl2ILeemJ9qoxHJCg,fasterrcnn_resnet50_fpn,0.3,0.315869,3.165870,1,0,0,10,11
3,_4SjTmQ-zn3XSv4D1-Tg4w,fasterrcnn_resnet50_fpn,0.3,0.419325,2.384782,14,0,0,16,30
4,_4ycMRRZcIpTeDT8MqtIDw,fasterrcnn_resnet50_fpn,0.3,0.508561,1.966332,0,0,0,13,13


In [27]:
# Create benchmark tables and model ranking

BENCHMARK_LONG = PREDICTION_COUNTS.merge(
    REFERENCE_DATA,
    on="image",
    how="left",
)

missing_reference = BENCHMARK_LONG[BENCHMARK_LONG["total_gt"].isna()]
if len(missing_reference) > 0:
    print(f"[WARNING] Missing reference data for {missing_reference['image'].nunique()} image(s).")

for category in CLASSES:
    BENCHMARK_LONG[f"{category}_signed_error"] = (
        BENCHMARK_LONG[f"{category}_pred"] - BENCHMARK_LONG[f"{category}_gt"]
    )
    BENCHMARK_LONG[f"{category}_abs_error"] = BENCHMARK_LONG[f"{category}_signed_error"].abs()
    BENCHMARK_LONG[f"{category}_undercount"] = np.maximum(
        BENCHMARK_LONG[f"{category}_gt"] - BENCHMARK_LONG[f"{category}_pred"],
        0,
    )
    BENCHMARK_LONG[f"{category}_overcount"] = np.maximum(
        BENCHMARK_LONG[f"{category}_pred"] - BENCHMARK_LONG[f"{category}_gt"],
        0,
    )
    BENCHMARK_LONG[f"{category}_relative_error"] = np.where(
        BENCHMARK_LONG[f"{category}_gt"] > 0,
        BENCHMARK_LONG[f"{category}_abs_error"] / BENCHMARK_LONG[f"{category}_gt"],
        np.where(BENCHMARK_LONG[f"{category}_pred"] == 0, 0, 1),
    )

BENCHMARK_LONG["total_signed_error"] = BENCHMARK_LONG["total_pred"] - BENCHMARK_LONG["total_gt"]
BENCHMARK_LONG["total_abs_error"] = BENCHMARK_LONG["total_signed_error"].abs()
BENCHMARK_LONG["total_undercount"] = np.maximum(
    BENCHMARK_LONG["total_gt"] - BENCHMARK_LONG["total_pred"],
    0,
)
BENCHMARK_LONG["total_overcount"] = np.maximum(
    BENCHMARK_LONG["total_pred"] - BENCHMARK_LONG["total_gt"],
    0,
)
BENCHMARK_LONG["total_relative_error"] = np.where(
    BENCHMARK_LONG["total_gt"] > 0,
    BENCHMARK_LONG["total_abs_error"] / BENCHMARK_LONG["total_gt"],
    np.where(BENCHMARK_LONG["total_pred"] == 0, 0, 1),
)

# Detection is the priority in this stage. Missing an object is worse than passing
# an extra crop to the later CNN categorisation module, so undercount is penalised
# more strongly than overcount.
BENCHMARK_LONG["detection_priority_error"] = (
    2.0 * BENCHMARK_LONG["total_undercount"] + BENCHMARK_LONG["total_overcount"]
)

BENCHMARK_LONG.to_csv(RESULTS_PATH / "benchmark_long.csv", index=False)

ranking_agg = {
    "images_count": ("image", "count"),
    "mean_detection_priority_error": ("detection_priority_error", "mean"),
    "mean_total_abs_error": ("total_abs_error", "mean"),
    "sum_total_abs_error": ("total_abs_error", "sum"),
    "mean_total_undercount": ("total_undercount", "mean"),
    "mean_total_overcount": ("total_overcount", "mean"),
    "mean_total_relative_error": ("total_relative_error", "mean"),
    "mean_inference_time_s": ("inference_time_s", "mean"),
    "mean_fps": ("fps", "mean"),
    "mean_gt_total": ("total_gt", "mean"),
    "mean_pred_total": ("total_pred", "mean"),
}

for category in CLASSES:
    ranking_agg[f"mean_{category}_abs_error"] = (f"{category}_abs_error", "mean")
    ranking_agg[f"mean_{category}_undercount"] = (f"{category}_undercount", "mean")
    ranking_agg[f"mean_{category}_overcount"] = (f"{category}_overcount", "mean")
    ranking_agg[f"mean_{category}_relative_error"] = (f"{category}_relative_error", "mean")

RANKING = BENCHMARK_LONG.groupby(["model", "threshold"]).agg(**ranking_agg).reset_index()

max_priority_error = RANKING["mean_detection_priority_error"].max()
max_abs_error = RANKING["mean_total_abs_error"].max()
max_fps = RANKING["mean_fps"].max()

RANKING["detection_score"] = 1 - (
    RANKING["mean_detection_priority_error"] / max_priority_error
    if max_priority_error > 0 else 0
)
RANKING["count_score"] = 1 - (RANKING["mean_total_abs_error"] / max_abs_error if max_abs_error > 0 else 0)
RANKING["speed_score"] = RANKING["mean_fps"] / max_fps if max_fps > 0 else 0

# Detection-first score: speed is intentionally only a tie-breaker.
RANKING["final_score"] = 0.9 * RANKING["detection_score"] + 0.1 * RANKING["speed_score"]
RANKING = RANKING.sort_values(
    ["final_score", "mean_detection_priority_error", "mean_total_abs_error"],
    ascending=[False, True, True],
)

RANKING.to_csv(RESULTS_PATH / "model_ranking.csv", index=False)

# Accuracy-first ranking ignores speed and sorts by the detection-stage objective.
DETECTION_ACCURACY_RANKING = RANKING.sort_values(
    ["mean_detection_priority_error", "mean_total_abs_error", "mean_total_undercount"],
    ascending=[True, True, True],
)
DETECTION_ACCURACY_RANKING.to_csv(RESULTS_PATH / "detection_accuracy_ranking.csv", index=False)

# Best threshold for each model according to the detection-first score.
BEST_THRESHOLD_PER_MODEL = RANKING.loc[
    RANKING.groupby("model")["final_score"].idxmax()
].sort_values("final_score", ascending=False)
BEST_THRESHOLD_PER_MODEL.to_csv(RESULTS_PATH / "best_threshold_per_model.csv", index=False)

# Class-level summary helps explain why a model is good or bad.
CATEGORY_ERROR_SUMMARY = BENCHMARK_LONG.groupby(["model", "threshold"])[
    [f"{category}_abs_error" for category in CLASSES]
    + [f"{category}_undercount" for category in CLASSES]
    + ["total_abs_error", "total_undercount", "total_overcount", "detection_priority_error"]
].mean().reset_index().sort_values("detection_priority_error")
CATEGORY_ERROR_SUMMARY.to_csv(RESULTS_PATH / "category_error_summary.csv", index=False)

# Wide table is useful for quick manual inspection: lower value means better count match.
BENCHMARK_WIDE = BENCHMARK_LONG.pivot_table(
    index=["model", "threshold"],
    columns="image",
    values="total_abs_error",
    aggfunc="first",
)
BENCHMARK_WIDE.to_csv(RESULTS_PATH / "benchmark_wide_total_abs_error.csv")

print("Saved benchmark files to:", RESULTS_PATH)
print("Detection-first ranking:")
display(RANKING.head(10))
print("Accuracy-only ranking:")
display(DETECTION_ACCURACY_RANKING.head(10))
print("Best threshold per model:")
display(BEST_THRESHOLD_PER_MODEL)
BENCHMARK_LONG.head()


Saved benchmark files to: /content/drive/MyDrive/colab/mapilary_test_data/results
Detection-first ranking:


,model,threshold,images_count,mean_detection_priority_error,mean_total_abs_error,sum_total_abs_error,mean_total_undercount,mean_total_overcount,mean_total_relative_error,mean_inference_time_s,...,mean_motorcycle_overcount,mean_motorcycle_relative_error,mean_vehicle_abs_error,mean_vehicle_undercount,mean_vehicle_overcount,mean_vehicle_relative_error,detection_score,count_score,speed_score,final_score
7,facebook/detr-resnet-50,0.5,20,5.40,4.90,98,0.50,4.40,0.728933,0.190122,...,0.0,0.0600,4.10,0.30,3.80,0.739671,0.782258,0.675497,0.574945,0.761527
8,facebook/detr-resnet-50,0.7,20,6.35,4.35,87,2.00,2.35,0.543102,0.219933,...,0.0,0.1300,3.65,1.35,2.30,0.552083,0.743952,0.711921,0.490509,0.718607
12,fasterrcnn_resnet50_fpn,0.3,20,6.75,6.10,122,0.65,5.45,0.951750,0.386410,...,0.0,0.1425,5.95,0.45,5.50,1.010185,0.727823,0.596026,0.281094,0.683150
21,retinanet_resnet50_fpn,0.3,20,7.25,6.20,124,1.05,5.15,0.745581,0.390171,...,0.0,0.1425,6.30,0.85,5.45,0.849868,0.707661,0.589404,0.280661,0.664961
0,PekingU/rtdetr_r18vd,0.3,20,8.70,5.55,111,3.15,2.40,0.566052,0.139562,...,0.0,0.1400,4.55,1.85,2.70,0.611480,0.649194,0.632450,0.793318,0.663606
18,microsoft/conditional-detr-resnet-50,0.3,20,8.10,4.90,98,3.20,1.70,0.502233,0.189812,...,0.0,0.1525,3.90,1.95,1.95,0.510605,0.673387,0.675497,0.566276,0.662676
24,retinanet_resnet50_fpn_v2,0.3,20,7.80,7.00,140,0.80,6.20,1.067434,0.354103,...,0.0,0.1200,7.20,0.40,6.80,1.150570,0.685484,0.536424,0.305201,0.647456
13,fasterrcnn_resnet50_fpn,0.5,20,8.30,5.35,107,2.95,2.40,0.652624,0.374384,...,0.0,0.1425,4.35,1.90,2.45,0.643406,0.665323,0.645695,0.294757,0.628266
3,SenseTime/deformable-detr,0.3,20,8.30,5.00,100,3.30,1.70,0.516501,0.410051,...,0.0,0.1525,3.70,2.05,1.65,0.504418,0.665323,0.668874,0.257949,0.624585
14,fasterrcnn_resnet50_fpn,0.7,20,10.10,5.45,109,4.65,0.80,0.472140,0.332834,...,0.0,0.1425,3.95,2.90,1.05,0.473577,0.592742,0.639073,0.327187,0.566186


Accuracy-only ranking:


,model,threshold,images_count,mean_detection_priority_error,mean_total_abs_error,sum_total_abs_error,mean_total_undercount,mean_total_overcount,mean_total_relative_error,mean_inference_time_s,...,mean_motorcycle_overcount,mean_motorcycle_relative_error,mean_vehicle_abs_error,mean_vehicle_undercount,mean_vehicle_overcount,mean_vehicle_relative_error,detection_score,count_score,speed_score,final_score
7,facebook/detr-resnet-50,0.5,20,5.40,4.90,98,0.50,4.40,0.728933,0.190122,...,0.0,0.0600,4.10,0.30,3.80,0.739671,0.782258,0.675497,0.574945,0.761527
8,facebook/detr-resnet-50,0.7,20,6.35,4.35,87,2.00,2.35,0.543102,0.219933,...,0.0,0.1300,3.65,1.35,2.30,0.552083,0.743952,0.711921,0.490509,0.718607
12,fasterrcnn_resnet50_fpn,0.3,20,6.75,6.10,122,0.65,5.45,0.951750,0.386410,...,0.0,0.1425,5.95,0.45,5.50,1.010185,0.727823,0.596026,0.281094,0.683150
21,retinanet_resnet50_fpn,0.3,20,7.25,6.20,124,1.05,5.15,0.745581,0.390171,...,0.0,0.1425,6.30,0.85,5.45,0.849868,0.707661,0.589404,0.280661,0.664961
24,retinanet_resnet50_fpn_v2,0.3,20,7.80,7.00,140,0.80,6.20,1.067434,0.354103,...,0.0,0.1200,7.20,0.40,6.80,1.150570,0.685484,0.536424,0.305201,0.647456
18,microsoft/conditional-detr-resnet-50,0.3,20,8.10,4.90,98,3.20,1.70,0.502233,0.189812,...,0.0,0.1525,3.90,1.95,1.95,0.510605,0.673387,0.675497,0.566276,0.662676
3,SenseTime/deformable-detr,0.3,20,8.30,5.00,100,3.30,1.70,0.516501,0.410051,...,0.0,0.1525,3.70,2.05,1.65,0.504418,0.665323,0.668874,0.257949,0.624585
13,fasterrcnn_resnet50_fpn,0.5,20,8.30,5.35,107,2.95,2.40,0.652624,0.374384,...,0.0,0.1425,4.35,1.90,2.45,0.643406,0.665323,0.645695,0.294757,0.628266
0,PekingU/rtdetr_r18vd,0.3,20,8.70,5.55,111,3.15,2.40,0.566052,0.139562,...,0.0,0.1400,4.55,1.85,2.70,0.611480,0.649194,0.632450,0.793318,0.663606
14,fasterrcnn_resnet50_fpn,0.7,20,10.10,5.45,109,4.65,0.80,0.472140,0.332834,...,0.0,0.1425,3.95,2.90,1.05,0.473577,0.592742,0.639073,0.327187,0.566186


Best threshold per model:


,model,threshold,images_count,mean_detection_priority_error,mean_total_abs_error,sum_total_abs_error,mean_total_undercount,mean_total_overcount,mean_total_relative_error,mean_inference_time_s,...,mean_motorcycle_overcount,mean_motorcycle_relative_error,mean_vehicle_abs_error,mean_vehicle_undercount,mean_vehicle_overcount,mean_vehicle_relative_error,detection_score,count_score,speed_score,final_score
7,facebook/detr-resnet-50,0.5,20,5.40,4.90,98,0.50,4.40,0.728933,0.190122,...,0.0,0.0600,4.10,0.30,3.80,0.739671,0.782258,0.675497,0.574945,0.761527
12,fasterrcnn_resnet50_fpn,0.3,20,6.75,6.10,122,0.65,5.45,0.951750,0.386410,...,0.0,0.1425,5.95,0.45,5.50,1.010185,0.727823,0.596026,0.281094,0.683150
21,retinanet_resnet50_fpn,0.3,20,7.25,6.20,124,1.05,5.15,0.745581,0.390171,...,0.0,0.1425,6.30,0.85,5.45,0.849868,0.707661,0.589404,0.280661,0.664961
0,PekingU/rtdetr_r18vd,0.3,20,8.70,5.55,111,3.15,2.40,0.566052,0.139562,...,0.0,0.1400,4.55,1.85,2.70,0.611480,0.649194,0.632450,0.793318,0.663606
18,microsoft/conditional-detr-resnet-50,0.3,20,8.10,4.90,98,3.20,1.70,0.502233,0.189812,...,0.0,0.1525,3.90,1.95,1.95,0.510605,0.673387,0.675497,0.566276,0.662676
24,retinanet_resnet50_fpn_v2,0.3,20,7.80,7.00,140,0.80,6.20,1.067434,0.354103,...,0.0,0.1200,7.20,0.40,6.80,1.150570,0.685484,0.536424,0.305201,0.647456
3,SenseTime/deformable-detr,0.3,20,8.30,5.00,100,3.30,1.70,0.516501,0.410051,...,0.0,0.1525,3.70,2.05,1.65,0.504418,0.665323,0.668874,0.257949,0.624585
9,fasterrcnn_mobilenet_v3_large_fpn,0.3,20,11.10,7.35,147,3.75,3.60,0.851418,0.271018,...,0.0,0.2025,6.20,2.75,3.45,0.872147,0.552419,0.513245,0.407819,0.537959
16,fcos_resnet50_fpn,0.5,20,12.50,6.65,133,5.85,0.80,0.449532,0.374817,...,0.0,0.1400,4.45,3.55,0.90,0.431931,0.495968,0.559603,0.290387,0.475410
27,ssd300_vgg16,0.3,20,19.90,9.95,199,9.95,0.00,0.664112,0.283542,...,0.0,0.1875,6.50,6.50,0.00,0.614975,0.197581,0.341060,0.389580,0.216781


,image,model,threshold,inference_time_s,fps,pedestrian_pred,bicycle_pred,motorcycle_pred,vehicle_pred,total_pred,...,vehicle_abs_error,vehicle_undercount,vehicle_overcount,vehicle_relative_error,total_signed_error,total_abs_error,total_undercount,total_overcount,total_relative_error,detection_priority_error
0,_1Gn_xkw7sa_i9GU4mkxxQ,fasterrcnn_resnet50_fpn,0.3,0.386411,2.587918,6,0,3,6,15,...,1,0,1,0.200000,5,5,0,5,0.500000,5.0
1,_2yFuPMD1eo4D1sHkx-c4Q,fasterrcnn_resnet50_fpn,0.3,0.336866,2.968542,2,0,0,6,8,...,1,0,1,0.200000,3,3,0,3,0.600000,3.0
2,_3JWmnl2ILeemJ9qoxHJCg,fasterrcnn_resnet50_fpn,0.3,0.315869,3.165870,1,0,0,10,11,...,2,0,2,0.250000,3,3,0,3,0.375000,3.0
3,_4SjTmQ-zn3XSv4D1-Tg4w,fasterrcnn_resnet50_fpn,0.3,0.419325,2.384782,14,0,0,16,30,...,1,1,0,0.058824,-5,5,5,0,0.142857,10.0
4,_4ycMRRZcIpTeDT8MqtIDw,fasterrcnn_resnet50_fpn,0.3,0.508561,1.966332,0,0,0,13,13,...,0,0,0,0.000000,0,0,0,0,0.000000,0.0


In [28]:
# Best model per image and threshold

BEST_PER_IMAGE = BENCHMARK_LONG.loc[
    BENCHMARK_LONG.groupby(["image", "threshold"])["total_abs_error"].idxmin(),
    ["image", "threshold", "model", "total_gt", "total_pred", "total_abs_error"],
].sort_values(["threshold", "image"])

BEST_PER_IMAGE.to_csv(RESULTS_PATH / "best_model_per_image.csv", index=False)

# This table is useful for checking whether one model wins consistently,
# or whether different images prefer different detectors.
BEST_PER_IMAGE.head(20)


,image,threshold,model,total_gt,total_pred,total_abs_error
160,_1Gn_xkw7sa_i9GU4mkxxQ,0.3,microsoft/conditional-detr-resnet-50,10,11,1
21,_2yFuPMD1eo4D1sHkx-c4Q,0.3,fasterrcnn_mobilenet_v3_large_fpn,5,4,1
162,_3JWmnl2ILeemJ9qoxHJCg,0.3,microsoft/conditional-detr-resnet-50,8,9,1
83,_4SjTmQ-zn3XSv4D1-Tg4w,0.3,fcos_resnet50_fpn,35,38,3
4,_4ycMRRZcIpTeDT8MqtIDw,0.3,fasterrcnn_resnet50_fpn,13,13,0
165,_6lYRZS5jTuojhqapjEL1Q,0.3,microsoft/conditional-detr-resnet-50,11,10,1
66,_92GTJv0x4fIix8B2hqahg,0.3,retinanet_resnet50_fpn_v2,15,15,0
187,_EGt7GTUW6OKw0zzz7c5-A,0.3,SenseTime/deformable-detr,6,6,0
108,_F2SFIm0YzTzrDuuDZP_BA,0.3,ssd300_vgg16,4,1,3
49,_G--4T8xsKtSDNBXVPDrxg,0.3,retinanet_resnet50_fpn,18,15,3


## Interpretacja wyników

Najważniejszym plikiem wynikowym jest `model_ranking.csv`. W tym etapie priorytetem jest jak najdokładniejsze wykrycie obiektów, ponieważ wykryte obiekty będą następnie przekazywane do modułu CNN odpowiedzialnego za kategoryzację. Szybkość inferencji jest uwzględniana jedynie jako metryka pomocnicza.

Ranking używa metryki `detection_priority_error`, w której niedowykrycia są karane silniej niż nadwykrycia. Wynika to z założenia, że obiekt pominięty przez detektor nie trafi już do dalszej kategoryzacji, natomiast nadmiarowa detekcja może zostać później odrzucona lub sklasyfikowana jako nieistotna.

Plik `detection_accuracy_ranking.csv` pokazuje ranking nastawiony wyłącznie na jakość detekcji, bez premiowania szybkości. Plik `benchmark_long.csv` zawiera pełne wyniki dla każdego obrazu, modelu i progu detekcji. Na jego podstawie można sprawdzić, które obrazy są najtrudniejsze oraz dla których klas model popełnia największe błędy.

Plik `category_error_summary.csv` pokazuje średni błąd osobno dla klas `pedestrian`, `bicycle`, `motorcycle` i `vehicle`. Jest to potrzebne, ponieważ niski błąd całkowity nie zawsze oznacza, że model dobrze radzi sobie ze wszystkimi kategoriami.

Wyniki benchmarku należy interpretować jako ocenę przydatności modelu do wykrywania i zliczania obiektów ruchu drogowego. Nie jest to pełna ewaluacja detekcji typu mAP/IoU, ponieważ notebook nie sprawdza dokładnego dopasowania ramek ograniczających do obiektów.
